#### Loading the forecast dataset

In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

In [ ]:
baseline_data = pd.read_csv(
    "../data/processed/cement_forecasting_model_data.csv",
    parse_dates=["date"]
)

baseline_data.shape

(32040, 22)

In [2]:
# Spliting the dataset
last_training_date = baseline_data["date"].max() - pd.Timedelta(days=56)

train_data = baseline_data[baseline_data["date"] <= last_training_date]
test_data = baseline_data[baseline_data["date"] > last_training_date]

In [3]:
# Checking the shape
train_data.shape, test_data.shape

((30360, 22), (1680, 22))

In [4]:
site = train_data["site_id"].unique()[0]

site_train = train_data[train_data["site_id"] == site]
site_test = test_data[test_data["site_id"] == site]

site, site_train.shape, site_test.shape

('SITE_001', (1012, 22), (56, 22))

In [5]:
# From the business problem we are defining the external variables as planned pour tonnes, rain  and avg temp

external_features = [
    "planned_pour_tonnes",
    "rain_mm",
    "avg_temp_c"
]

y_train = site_train["consumed_tonnes"]
X_train = site_train[external_features]

y_test = site_test["consumed_tonnes"]
X_test = site_test[external_features]

### Modelling

### Baseline model - SARIMAX

In [23]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

In [15]:
baseline_model = SARIMAX(
    y_train.reset_index(drop=True),
    exog=X_train.reset_index(drop=True),
    order=(1, 0, 0),
    seasonal_order=(1, 0, 0, 7),
    trend="c"
)

baseline_result = baseline_model.fit(disp=False)

print("Baseline model fitted successfully.")

Baseline model fitted successfully.


In [16]:
#Making predictions for the 56 days which is 8 weeks
predictions = baseline_result.forecast(
    steps=56,
    exog=X_test
)

predictions.head()

1012    31.769595
1013    29.032557
1014    33.794267
1015    33.721062
1016    39.260082
Name: predicted_mean, dtype: float64

In [24]:
# Evaluation of the baseline mdoel
mape = mean_absolute_percentage_error(y_test, predictions) * 100
rmse = mean_squared_error(y_test, predictions) ** 0.5

print("MAPE:", round(mape, 2), "%")
print("RMSE:", round(rmse, 2), "tonnes")

MAPE: 6.147076730574193e+17 %
RMSE: 10.32 tonnes


- The MAPE: 6.147076730574193e+17 % seems to be invalide because where actual-demand values are 0. MAPE divides the prediction error by actual demand, and division by zero produces an enormous number. 

- Let's resolve this issue by calculating the MAPE where actual demand > 0


In [25]:
actual = y_test.to_numpy()
predicted = predictions.to_numpy()

non_zero = actual > 0

mape = mean_absolute_percentage_error(
    actual[non_zero],
    predicted[non_zero]
) * 100

print("MAPE:", round(mape, 2), "%")

MAPE: 29.06 %


In [ ]:
#Let make the focus for the all sites
all_forecasts = []

for current_site in train_data["site_id"].unique():

    current_train = train_data[
        train_data["site_id"] == current_site
    ]

    current_test = test_data[
        test_data["site_id"] == current_site
    ]

    model = SARIMAX(
        current_train["consumed_tonnes"],
        exog=current_train[external_features],
        order=(1, 0, 0),
        seasonal_order=(1, 0, 0, 7),
        trend="c"
    )

    fitted_model = model.fit(disp=False)

    current_predictions = fitted_model.forecast(
        steps=56,
        exog=current_test[external_features]
    )

    result = current_test[
        ["date", "site_id", "consumed_tonnes"]
    ].copy()

    result["predicted_tonnes"] = current_predictions.to_numpy()

    all_forecasts.append(result)

In [27]:
baseline_forecasts = pd.concat(all_forecasts, ignore_index=True)

baseline_forecasts.shape

(1680, 4)

In [28]:
actual = baseline_forecasts["consumed_tonnes"]
predicted = baseline_forecasts["predicted_tonnes"]

non_zero = actual > 0

overall_mape = mean_absolute_percentage_error(
    actual[non_zero],
    predicted[non_zero]
) * 100

overall_rmse = mean_squared_error(
    actual,
    predicted
) ** 0.5

print("Overall MAPE:", round(overall_mape, 2), "%")
print("Overall RMSE:", round(overall_rmse, 2), "tonnes")

Overall MAPE: 23.79 %
Overall RMSE: 9.16 tonnes


In [30]:
# Saving the baseline forecasts
baseline_forecasts.to_csv(
    "../outputs/sarimax_baseline_forecasts.csv",
    index=False
)